# 03 · Personalized anomaly detection

Does a night differ from the same person's earlier nights? The detector was fixed in
`docs/decisions.md` (41–45) and committed before any run.

- **Baseline.** Expected heart rate and movement for each person, sleep stage and time of night,
  built from that person's earlier nights and shrunk toward a population prior. With few nights,
  a person looks like the population.
- **Night channels**, all in the "worse sleep" direction:
  - the highest 30-min heart-rate z;
  - the whole-night heart-rate z;
  - wake bouts per hour;
  - onset latency.
- **Alarm.** A night is flagged when its smallest channel tail probability falls under a
  threshold. The threshold is set on *other people's* clean nights, scored out of sample, for a
  5% false-alarm rate.
- **Validation.** Anomalies are injected into the raw signals of test nights at known times and
  sizes, then pushed through the same feature, staging and scoring code.
- **Primary detector:** stages predicted by the population GRU (what a real user would have),
  with the personal baseline. Expert stages are the best case, and "none" is a stage-free
  detector. Each also runs with a population-only baseline.

Nothing in this notebook is fitted or tuned; it only reads the saved run. There are no causal
claims: a flag means "different from this person's earlier nights", not why.

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sleepwatch.anomaly import baseline as bl
from sleepwatch.anomaly import scores as sc
from sleepwatch.anomaly.experiment import AnomalyConfig, _inputs
from sleepwatch.config import get_settings
from sleepwatch.constants import EPOCH_S
from sleepwatch.features.build import load_build
from sleepwatch.models.data import prepare
from sleepwatch.models.splits import load_folds

settings = get_settings()
index = pd.read_csv(settings.results_dir / "anomaly_index.csv")
latest = index[~index["partial"]].iloc[-1]
run_dir = settings.results_dir / latest["name"] / latest["run"]
info = json.loads((run_dir / "run.json").read_text())
cfg = AnomalyConfig.model_validate(
    __import__("yaml").safe_load((run_dir / "config.yaml").read_text())
)
metrics = json.loads((run_dir / "metrics.json").read_text())
scores = pd.read_parquet(run_dir / "night_scores.parquet")
for column in ["window", "bouts", "window_stages"]:
    scores[column] = scores[column].map(lambda v: json.loads(v) if isinstance(v, str) else None)
truth = pd.read_parquet(run_dir / "injections.parquet")
truth["windows"] = truth["windows"].map(json.loads)
epochs_run = pd.read_parquet(run_dir / "epochs.parquet")
stages = pd.read_parquet(run_dir / "stages.parquet")
nights = pd.read_parquet(run_dir / "nights.parquet")
flagged = json.loads((run_dir / "flagged_nights.json").read_text())
print(f"run {latest['run']} at commit {info['git']['commit'][:8]}; dirty: {info['git']['dirty']}")
assert not info["git"]["dirty"] and not latest["partial"]
print("integrity checks:", info["checks"])
assert (
    info["checks"]["features_equal"] == info["checks"]["stages_equal"] == info["checks"]["nights"]
)
primary = scores[(scores["source"] == "gru") & (scores["baseline"] == "personal")]
tested = primary[primary["version"] == "clean"]
print(
    f"{int(nights['scored'].sum())} scored nights ({len(tested)} as test nights), "
    f"{nights['subject'].nunique()} subjects; excluded: {cfg.exclude}"
)
per_night = truth.groupby(["subject", "night"]).size()
skipped = sorted(
    set(map(tuple, tested[["subject", "night"]].to_numpy()))
    - set(map(tuple, truth.query("kind != 'hr'")[["subject", "night"]].to_numpy()))
)
print(
    f"injected versions per night: {per_night.min()}-{per_night.max()} "
    f"({len(truth)} in total); nights without a wake stretch to copy: {skipped}"
)

COLORS = {
    "gru": "#2a78d6",
    "expert": "#eb6834",
    "none": "#a9a8a3",
    "hr": "#2a78d6",
    "frag": "#eb6834",
    "onset": "#7e57c2",
}
plt.rcParams.update(
    {
        "figure.facecolor": "#fcfcfb",
        "axes.facecolor": "#fcfcfb",
        "axes.edgecolor": "#a9a8a3",
        "axes.labelcolor": "#52514e",
        "xtick.color": "#52514e",
        "ytick.color": "#52514e",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.color": "#e6e5e1",
        "grid.linewidth": 0.6,
        "lines.linewidth": 1.2,
        "axes.titlesize": 11,
        "axes.titleweight": "bold",
        "font.size": 9,
        "legend.frameon": False,
    }
)


def ci(value, bounds, digits=2):
    return f"{value:.{digits}f} [{bounds[0]:.{digits}f}, {bounds[1]:.{digits}f}]"

## 1 · What the detector sees

Each panel shows a night's heart rate against the personal expectation (±1 SD band) and the
predicted hypnogram. The expectation is recomputed here with the library functions, from the
saved stages and the stored epoch table, and checked against the saved channel values. The
first panel is a clean test night with a typical score; the others are injected versions of the
same night (shaded: where the anomaly was put).

In [ ]:
epochs, quality, _ = load_build(cfg.features, settings.processed_dir)
data = prepare(epochs)
folds = load_folds()
usable = nights[nights["usable"]]
rank = nights.set_index(["subject", "night"])


def detector_view(subject, night, version):
    """Inputs, expected HR and z for one night version, as the primary detector saw it."""
    fold = folds[subject]
    st = stages[stages["fold"] == fold]
    keys = ["subject", "night", "epoch"]
    clean = data.merge(usable[["subject", "night"]], on=["subject", "night"])
    train = clean.merge(st[st["role"] == "train"][[*keys, "gru"]], on=keys)
    prior = bl.fit_prior(_inputs(train, "gru"))
    mine = clean[clean["subject"] == subject].merge(
        st[st["role"] == "test"][[*keys, "gru"]], on=keys
    )
    history = _inputs(mine, "gru")
    this = rank.loc[(subject, night), "night_rank"]
    earlier = [
        (subject, n)
        for (s, n), r in rank.iterrows()
        if s == subject and r["usable"] and r["night_rank"] < this
    ]
    offsets = bl.personal_offsets(bl.night_summaries(history, prior).loc[earlier], prior)
    target = epochs_run[
        (epochs_run["subject"] == subject)
        & (epochs_run["night"] == night)
        & (epochs_run["version"] == version)
    ].sort_values("epoch")
    view = _inputs(target.reset_index(drop=True), "gru")
    view = view.join(bl.score_epochs(view, prior, offsets))
    view["sd_hr"] = prior.sd(view, "hr")
    population = bl.score_epochs(view, prior, None)
    ch = sc.night_channels(view.assign(z_act_pop=population["z_act"]), True, cfg.channels)
    saved = primary[
        (primary["subject"] == subject)
        & (primary["night"] == night)
        & (primary["version"] == version)
    ].iloc[0]
    assert np.isclose(ch["hr_window"], saved["hr_window"], equal_nan=True), "recomputation differs"
    assert np.isclose(ch["hr_night"], saved["hr_night"], equal_nan=True)
    return view, saved


def plot_night(ax_hr, ax_stage, view, title, shade=(), flag=None):
    hours = view["epoch"] * EPOCH_S / 3600
    ax_hr.fill_between(
        hours,
        view["expected_hr"] - view["sd_hr"],
        view["expected_hr"] + view["sd_hr"],
        color="#cde2fb",
        lw=0,
        label="expected ±1 SD",
    )
    ax_hr.plot(hours, view["hr"], color="#0b0b0b", lw=0.8, label="heart rate")
    for a, b in shade:
        ax_hr.axvspan(a * EPOCH_S / 3600, b * EPOCH_S / 3600, color="#eb6834", alpha=0.18, lw=0)
    if flag is not None:
        ax_hr.axvspan(
            flag[0] * EPOCH_S / 3600,
            flag[1] * EPOCH_S / 3600,
            fill=False,
            ec="#7e57c2",
            lw=1.2,
            label="highest 30-min window",
        )
    ax_hr.set(title=title, ylabel="bpm")
    ax_stage.step(
        hours,
        view["stage"].map({0: 4, 4: 3, 1: 2, 2: 1, 3: 0}),
        where="post",
        color="#2a78d6",
        lw=0.8,
    )
    ax_stage.set_yticks([0, 1, 2, 3, 4], ["N3", "N2", "N1", "REM", "Wake"])
    ax_stage.set(xlabel="hours since recording start")
    ax_stage.grid(False)


typical = tested.iloc[(tested["hr_night"] - tested["hr_night"].median()).abs().argsort()].iloc[0]
subject, night = typical["subject"], int(typical["night"])
examples = ["clean", "hr+6_120", "frag4", "onset+40"]
fig, axes = plt.subplots(
    8, 1, figsize=(11, 15), gridspec_kw={"height_ratios": [3, 1] * 4}, sharex=True
)
for i, version in enumerate(examples):
    if version not in set(epochs_run.query("subject == @subject and night == @night")["version"]):
        continue
    view, saved = detector_view(subject, night, version)
    shade = []
    if version != "clean":
        shade = truth.query("subject == @subject and night == @night and version == @version")[
            "windows"
        ].iloc[0]
    flag_text = "flagged" if saved["flagged"] else "not flagged"
    plot_night(
        axes[2 * i],
        axes[2 * i + 1],
        view,
        f"{subject}/{night} · {version} · {flag_text} (top channel: {saved['top_channel']})",
        shade,
        saved["window"],
    )
axes[0].legend(fontsize=8, loc="upper right")
plt.tight_layout()
plt.show()

## 2 · False alarms

The alarm threshold was set on training subjects' clean nights so that at most 5% would be
flagged. The question is whether that carries over to **new people**: the false-alarm rate on
the clean test nights. A rate well above 5% would mean the calibration doesn't transfer.

In [ ]:
rows = []
for name, entry in metrics["variants"].items():
    fa = entry["false_alarm_rate"]
    source, variant = name.split("/")
    part = scores[(scores["source"] == source) & (scores["baseline"] == variant)]
    rows.append(
        {
            "stages": source,
            "baseline": variant,
            "false-alarm rate on clean test nights [95% CI]": ci(fa["value"], fa["ci95"], 3),
            "clean test nights": fa["nights"],
            "null rate on training subjects (by fold)": ", ".join(
                f"{r:.3f}" for r in part.groupby("fold")["null_rate"].first()
            ),
        }
    )
display(pd.DataFrame(rows))

## 3 · Detection of injected anomalies (primary detector)

- **Recall:** the share of injected nights flagged at the calibrated threshold.
- **AUROC:** how well the continuous score separates injected nights from clean test nights,
  regardless of threshold.
- **Precision:** derived for an *assumed* 10% of nights being anomalous, from recall and the
  observed false-alarm rate.
- **Localized:** for 30-min and 2-h HR injections, the share of flagged nights whose highest
  window is centred inside the injected window.
- **Awakenings seen:** the share of inserted awakenings that overlap a wake bout in the predicted
  stages.
- **Right channel:** the share of flagged nights whose smallest tail probability is in the
  channel matching the injected type.

95% CIs resample subjects.

In [ ]:
def condition_table(name):
    rows = []
    for version, c in metrics["variants"][name]["conditions"].items():
        rows.append(
            {
                "version": version,
                "kind": c["kind"],
                "size": c["size"],
                "duration (min)": c["duration_min"],
                "nights": c["nights"],
                "recall [95% CI]": ci(c["recall"], c["recall_ci95"]),
                "AUROC [95% CI]": ci(c["auroc"], c["auroc_ci95"]),
                "precision @10%": round(c["precision_at_prevalence"], 2),
                "localized": round(c["localized"], 2) if "localized" in c else None,
                "awakenings seen": round(c["awakenings_seen"], 2)
                if "awakenings_seen" in c
                else None,
                "right channel": round(c["type_correct"], 2)
                if c["type_correct"] == c["type_correct"]
                else None,
                "_recall": c["recall"],
                "_low": c["recall_ci95"][0],
                "_high": c["recall_ci95"][1],
            }
        )
    return pd.DataFrame(rows)


table = condition_table("gru/personal")
display(table.drop(columns=["_recall", "_low", "_high"]))

fig, axes = plt.subplots(1, 3, figsize=(12, 3.4), sharey=True)
hr = table[table["kind"] == "hr"]
for duration, color in [(30, "#a9a8a3"), (120, "#2a78d6"), (0, "#0d366b")]:
    part = hr[hr["duration (min)"] == duration].sort_values("size")
    label = "whole sleep period" if duration == 0 else f"{duration} min"
    axes[0].errorbar(
        part["size"],
        part["_recall"],
        yerr=[part["_recall"] - part["_low"], part["_high"] - part["_recall"]],
        marker="o",
        capsize=3,
        color=color,
        label=label,
    )
axes[0].set(title="HR elevation", xlabel="+ bpm", ylabel="recall at 5% false alarms")
axes[0].legend(fontsize=8)
for ax, kind, xlabel in [(axes[1], "frag", "awakenings of 3 min"), (axes[2], "onset", "+ min")]:
    part = table[table["kind"] == kind].sort_values("size")
    ax.errorbar(
        part["size"],
        part["_recall"],
        yerr=[part["_recall"] - part["_low"], part["_high"] - part["_recall"]],
        marker="o",
        capsize=3,
        color=COLORS[kind],
    )
    ax.set(title={"frag": "Fragmentation", "onset": "Delayed onset"}[kind], xlabel=xlabel)
for ax in axes:
    ax.axhline(
        metrics["variants"]["gru/personal"]["false_alarm_rate"]["value"],
        color="#52514e",
        lw=1,
        ls="--",
    )
    ax.set_ylim(0, 1.02)
plt.tight_layout()
plt.show()

## 4 · What the personal baseline and the stages add

Recall per condition for all six detectors, then the paired recall differences of the primary
detector against three alternatives: a population-only baseline, expert stages and no stages.
The baseline comparison is the direct test of personalization in Phase 3.

In [ ]:
recall = pd.DataFrame(
    {
        name: {v: c["recall"] for v, c in entry["conditions"].items()}
        for name, entry in metrics["variants"].items()
    }
)
display(recall.round(2))
diffs = []
for label, comparison in metrics["comparisons"].items():
    for version, d in comparison.items():
        diffs.append(
            {
                "comparison": label,
                "version": version,
                "recall difference [95% CI]": ci(d["recall_difference"], d["ci95"]),
            }
        )
display(
    pd.DataFrame(diffs).pivot(
        index="version", columns="comparison", values="recall difference [95% CI]"
    )
)

## 5 · Does a longer baseline help?

Recall of the primary detector by the number of earlier nights the baseline had. Descriptive
only: the groups contain different people.

In [ ]:
by_n = pd.DataFrame({k: v["recall"] for k, v in metrics["by_baseline_nights"].items()})
by_n.loc["false-alarm rate"] = [
    v["false_alarm_rate"] for v in metrics["by_baseline_nights"].values()
]
by_n.loc["nights"] = [v["nights"] for v in metrics["by_baseline_nights"].values()]
display(by_n.round(2))

## 6 · Real nights the detector flags

These are clean test nights flagged by the primary detector. About 5% of nights are expected to
be flagged by chance alone, so a flag is a prompt to look, not a finding. The table gives the
structured summary passed to the Phase 4 agent; the plots show where the night departs from the
personal expectation. Known data problems are listed where they apply. **No causes are
inferred.**

In [ ]:
print(
    f"{len(flagged)} of {len(tested)} clean test nights flagged "
    f"(expected by chance at 5%: about {0.05 * len(tested):.0f})"
)
summary = pd.DataFrame(
    [
        {
            "night": f"{f['subject']}/{f['night']}",
            "date": f["date_local"],
            "baseline nights": f["baseline_nights"],
            "top channel": f["top_channel"],
            "fired": ", ".join(c["channel"] for c in f["channels"] if c["fired"]),
            "window": "-".join(
                (f[k] or "")[11:] for k in ("window_start_local", "window_end_local")
            ),
            "HR above expected (bpm)": (
                None if f["hr_excess_bpm"] is None else round(f["hr_excess_bpm"], 1)
            ),
            "window stages": ", ".join(f"{k} {v:.0%}" for k, v in f["window_stage_share"].items()),
            "wake bouts/h": None
            if f["wake_bouts_per_hour"] is None
            else round(f["wake_bouts_per_hour"], 1),
            "onset (min)": None
            if f["onset_latency_min"] is None
            else round(f["onset_latency_min"]),
            "data quality": "; ".join(f["data_quality"]),
        }
        for f in flagged
    ]
)
display(summary)
shown = flagged[:6]
if shown:
    fig, axes = plt.subplots(
        2 * len(shown),
        1,
        figsize=(11, 3.4 * len(shown)),
        gridspec_kw={"height_ratios": [3, 1] * len(shown)},
    )
    for i, f in enumerate(shown):
        view, saved = detector_view(f["subject"], f["night"], "clean")
        plot_night(
            axes[2 * i],
            axes[2 * i + 1],
            view,
            f"{f['subject']}/{f['night']} · fired: "
            + ", ".join(c["channel"] for c in f["channels"] if c["fired"]),
            flag=saved["window"],
        )
    axes[0].legend(fontsize=8, loc="upper right")
    plt.tight_layout()
    plt.show()